# Train 7.3M ELT Supervised Model (Kaggle T4x2)

This notebook pulls the `elt-supervised-7.3m` repository and begins training using the dual T4 GPUs available on Kaggle. It is pre-configured with optimizations for T4 hardware (e.g. `use_bfloat16: False` to avoid slow emulation).

**Persistent cache:** Checkpoints and latent files are backed up to `/kaggle/working/elt_cache/` so they survive `git clone` re-runs. If you attach a previous notebook output as a Kaggle input dataset, it will also restore from `/kaggle/input/`.

In [ ]:
!pip install torch torchvision diffusers accelerate tqdm pillow kagglehub

In [ ]:
import os, shutil, glob

if os.path.exists('/kaggle/working'):
    os.chdir('/kaggle/working')

CACHE_DIR = '/kaggle/working/elt_cache'
REPO_DIR  = '/kaggle/working/elt-supervised-7.3m'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(f'{CACHE_DIR}/checkpoints', exist_ok=True)

# ── STEP 1: Backup checkpoints & latents BEFORE deleting repo ──
if os.path.exists(f'{REPO_DIR}/checkpoints'):
    for f in glob.glob(f'{REPO_DIR}/checkpoints/*.pt'):
        dst = f'{CACHE_DIR}/checkpoints/{os.path.basename(f)}'
        print(f'  Backing up {os.path.basename(f)} -> cache')
        shutil.copy2(f, dst)

if os.path.exists(f'{REPO_DIR}/latents_ffhq_128.pt'):
    print('  Backing up latents_ffhq_128.pt -> cache')
    shutil.copy2(f'{REPO_DIR}/latents_ffhq_128.pt', f'{CACHE_DIR}/latents_ffhq_128.pt')

# ── STEP 2: Fresh clone ──
!rm -rf elt-supervised-7.3m
!git clone https://github.com/ParthBrijpuria/elt-supervised-7.3m.git
%cd elt-supervised-7.3m

# ── STEP 3: Restore from cache (same session) ──
os.makedirs('checkpoints', exist_ok=True)
for f in glob.glob(f'{CACHE_DIR}/checkpoints/*.pt'):
    dst = f'checkpoints/{os.path.basename(f)}'
    if not os.path.exists(dst):
        print(f'  Restoring {os.path.basename(f)} from cache')
        shutil.copy2(f, dst)

if os.path.exists(f'{CACHE_DIR}/latents_ffhq_128.pt') and not os.path.exists('latents_ffhq_128.pt'):
    print('  Restoring latents_ffhq_128.pt from cache')
    shutil.copy2(f'{CACHE_DIR}/latents_ffhq_128.pt', 'latents_ffhq_128.pt')

# ── STEP 4: Restore from /kaggle/input/ (cross-session, if attached as dataset) ──
for input_dir in glob.glob('/kaggle/input/*/checkpoints'):
    for f in glob.glob(f'{input_dir}/*.pt'):
        dst = f'checkpoints/{os.path.basename(f)}'
        if not os.path.exists(dst):
            print(f'  Restoring {os.path.basename(f)} from /kaggle/input/')
            shutil.copy2(f, dst)

for input_dir in glob.glob('/kaggle/input/*'):
    src = f'{input_dir}/latents_ffhq_128.pt'
    if os.path.exists(src) and not os.path.exists('latents_ffhq_128.pt'):
        print('  Restoring latents_ffhq_128.pt from /kaggle/input/')
        shutil.copy2(src, 'latents_ffhq_128.pt')

# ── Summary ──
existing_ckpts = glob.glob('checkpoints/*.pt')
has_latents = os.path.exists('latents_ffhq_128.pt')
print(f'\n✅ Ready! Checkpoints found: {len(existing_ckpts)}, Latent file: {"YES" if has_latents else "NO"}')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("greatgamedota/ffhq-face-data-set")

print("Path to dataset files:", path)

## Step 1: Pre-encode all images through the VAE (runs ONCE, ~3 min)

This encodes all 70,000 FFHQ images into VAE latent tensors and saves them to a `.pt` file.
Training then loads directly from RAM — **no VAE, no PNG decoding, no disk I/O** during training.

**Automatically skipped if `latents_ffhq_128.pt` already exists** (restored from cache).

In [ ]:
import os
if not os.path.exists('latents_ffhq_128.pt'):
    !python preprocess_latents.py --train_dir $path --output latents_ffhq_128.pt --batch_size 128
    # Backup to cache immediately
    import shutil
    shutil.copy2('latents_ffhq_128.pt', '/kaggle/working/elt_cache/latents_ffhq_128.pt')
    print('Backed up latents to cache.')
else:
    print('latents_ffhq_128.pt already exists, skipping preprocessing.')

## Step 2: Train with pre-encoded latents on dual T4 GPUs

Training auto-resumes from the latest checkpoint if one exists.

In [ ]:
!accelerate launch --num_processes 2 run_train.py --train_dir $path --latent_file latents_ffhq_128.pt --gpu_config gpu_config.json --epochs 1000 --batch_size 128

## Step 3: Backup checkpoints to cache (run after training or before stopping)

This ensures your checkpoints survive the next `git clone`.

In [ ]:
import shutil, glob, os
os.makedirs('/kaggle/working/elt_cache/checkpoints', exist_ok=True)
for f in glob.glob('checkpoints/*.pt'):
    shutil.copy2(f, f'/kaggle/working/elt_cache/checkpoints/{os.path.basename(f)}')
    print(f'Backed up {os.path.basename(f)}')
print('Done! Checkpoints are safe in /kaggle/working/elt_cache/')